In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2019_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2019-01.csv", sep=";")

In [3]:
df_2019_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2019-02.csv", sep=";")

In [4]:
# Concatenando os dataframes de 2019_01 a 2019_02

df_2019 = pd.concat((df_2019_01, df_2019_02))

In [5]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1004029 entries, 0 to 507298
Data columns (total 16 columns):
 #   Column             Non-Null Count    Dtype 
---  ------             --------------    ----- 
 0   Regiao - Sigla     1004029 non-null  object
 1   Estado - Sigla     1004029 non-null  object
 2   Municipio          1004029 non-null  object
 3   Revenda            1004029 non-null  object
 4   CNPJ da Revenda    1004029 non-null  object
 5   Nome da Rua        1004029 non-null  object
 6   Numero Rua         1003417 non-null  object
 7   Complemento        261785 non-null   object
 8   Bairro             1000240 non-null  object
 9   Cep                1004029 non-null  object
 10  Produto            1004029 non-null  object
 11  Data da Coleta     1004029 non-null  object
 12  Valor de Venda     1004029 non-null  object
 13  Valor de Compra    381869 non-null   object
 14  Unidade de Medida  1004029 non-null  object
 15  Bandeira           1004029 non-null  object
dtypes: obj

In [6]:
# Filtro aplicado na extração: apenas registros do estado de Pernambuco (UF == "PE")
# Decisão: reduzir volume de armazenamento e escopo do projeto
df_2019_pe = df_2019[df_2019["Estado - Sigla"] == "PE"]

In [7]:
# Ajustndo o tipo de dado da coluna "Data da Coleta" para datetime
df_2019_pe["Data da Coleta"] = pd.to_datetime(df_2019_pe["Data da Coleta"], errors="coerce")


/tmp/ipykernel_1520/470632786.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2019_pe["Data da Coleta"] = pd.to_datetime(df_2019_pe["Data da Coleta"], errors="coerce")


In [8]:
# Ajustando o tipo de dado da coluna "Valor de Venda" e "Valor de Compra" para numérico
df_2019_pe["Valor de Venda"] = pd.to_numeric(df_2019_pe["Valor de Venda"], errors="coerce")
df_2019_pe["Valor de Compra"] = pd.to_numeric(df_2019_pe["Valor de Compra"], errors="coerce")


/tmp/ipykernel_1520/4089498631.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2019_pe["Valor de Venda"] = pd.to_numeric(df_2019_pe["Valor de Venda"], errors="coerce")
/tmp/ipykernel_1520/4089498631.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2019_pe["Valor de Compra"] = pd.to_numeric(df_2019_pe["Valor de Compra"], errors="coerce")


In [9]:
df_2019_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
3345,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,GASOLINA,2019-02-01,NaN,NaN,R$ / litro,RAIZEN
3346,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,ETANOL,2019-02-01,NaN,NaN,R$ / litro,RAIZEN
3347,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,DIESEL,2019-02-01,NaN,NaN,R$ / litro,RAIZEN
3348,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,DIESEL S10,2019-02-01,NaN,NaN,R$ / litro,RAIZEN
3349,NE,PE,ARARIPINA,POSTO TREVO COMBUSTIVEIS EIRELI,07.920.417/0001-11,AVENIDA FLORENTINO ALVES BATISTA,910,NaN,CENTRO,56280-000,GASOLINA,2019-02-01,NaN,NaN,R$ / litro,DISLUB


In [10]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Bronze no BigQuery
table_id = os.getenv("BRONZE_2019")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [11]:
# Criação e carga da tabela de BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2019_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
